# 2026 World Cup Winner Simulation

This notebook trains a match outcome model from historical results and Elo ratings, draws a 48-team 2026 tournament, simulates the group stage plus knockout rounds, and estimates title odds across repeated tournament runs.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

from dataclasses import dataclass
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 20)
np.random.seed(2026)


In [ ]:
BASE_DIR = Path.cwd()
RESULTS_PATH = BASE_DIR / 'results.csv'
ELO_PATH = BASE_DIR / 'eloratings.csv'
SEED = 2026
SIMULATION_RUNS = 100
GROUP_NAMES = [chr(ord('A') + i) for i in range(12)]
HOSTS = ['United States', 'Mexico', 'Canada']

TEAM_CONFEDS = {
    'France': 'UEFA', 'England': 'UEFA', 'Germany': 'UEFA', 'Spain': 'UEFA',
    'Portugal': 'UEFA', 'Italy': 'UEFA', 'Netherlands': 'UEFA', 'Belgium': 'UEFA',
    'Croatia': 'UEFA', 'Denmark': 'UEFA', 'Switzerland': 'UEFA', 'Austria': 'UEFA',
    'Serbia': 'UEFA', 'Poland': 'UEFA', 'Turkey': 'UEFA', 'Sweden': 'UEFA',
    'Morocco': 'CAF', 'Senegal': 'CAF', 'Egypt': 'CAF', 'Nigeria': 'CAF',
    'Algeria': 'CAF', 'Tunisia': 'CAF', 'Ivory Coast': 'CAF', 'Cameroon': 'CAF',
    'Ghana': 'CAF', 'Argentina': 'CONMEBOL', 'Brazil': 'CONMEBOL', 'Uruguay': 'CONMEBOL',
    'Colombia': 'CONMEBOL', 'Chile': 'CONMEBOL', 'Peru': 'CONMEBOL', 'Japan': 'AFC',
    'South Korea': 'AFC', 'Iran': 'AFC', 'Saudi Arabia': 'AFC', 'Australia': 'AFC',
    'Qatar': 'AFC', 'Iraq': 'AFC', 'United Arab Emirates': 'AFC',
    'United States': 'CONCACAF', 'Mexico': 'CONCACAF', 'Canada': 'CONCACAF',
    'Costa Rica': 'CONCACAF', 'Jamaica': 'CONCACAF', 'Panama': 'CONCACAF',
    'New Zealand': 'OFC', 'Honduras': 'Playoff', 'Mali': 'Playoff'
}

TEAM_NAME_ALIASES = {
    'Ivory Coast': 'Ivory Coast',
    'South Korea': 'South Korea',
    'Saudi Arabia': 'Saudi Arabia',
    'United Arab Emirates': 'United Arab Emirates',
    'United States': 'United States',
    'Costa Rica': 'Costa Rica',
    'New Zealand': 'New Zealand',
}

print(f'Teams in tournament: {len(TEAM_CONFEDS)}')


In [ ]:
@dataclass
class MatchResult:
    home_team: str
    away_team: str
    home_goals: int
    away_goals: int
    winner: str | None


def normalize_team_name(name: str) -> str:
    return str(name).replace(' ', ' ').strip()


def build_name_lookup(names):
    return {normalize_team_name(name): name for name in names}


def load_data():
    results = pd.read_csv(RESULTS_PATH)
    elo = pd.read_csv(ELO_PATH)

    results['date'] = pd.to_datetime(results['date'], format='mixed')
    elo['date'] = pd.to_datetime(elo['date'], format='mixed')

    results['home_team'] = results['home_team'].map(normalize_team_name)
    results['away_team'] = results['away_team'].map(normalize_team_name)
    elo['team'] = elo['team'].map(normalize_team_name)

    return results, elo


def get_latest_elo(elo):
    latest = elo.sort_values('date').groupby('team', as_index=False).tail(1).copy()
    latest = latest[['team', 'rating', 'date']].rename(columns={'rating': 'elo'})
    return latest.sort_values('elo', ascending=False).reset_index(drop=True)


def prepare_training_data(results, elo):
    results = results.loc[
        results['date'] >= elo['date'].min(),
        ['date', 'home_team', 'away_team', 'home_score', 'away_score', 'neutral', 'tournament']
    ].copy()

    results['neutral'] = results['neutral'].astype(int)
    results['home_lookup'] = results['home_team']
    results['away_lookup'] = results['away_team']

    elo_home = elo.sort_values(['date', 'team']).rename(columns={'team': 'home_lookup', 'rating': 'home_elo'})
    elo_away = elo.sort_values(['date', 'team']).rename(columns={'team': 'away_lookup', 'rating': 'away_elo'})

    merged = pd.merge_asof(
        results.sort_values(['date', 'home_lookup']),
        elo_home[['date', 'home_lookup', 'home_elo']].sort_values(['date', 'home_lookup']),
        on='date',
        by='home_lookup',
        direction='backward',
        allow_exact_matches=False,
    )

    merged = pd.merge_asof(
        merged.sort_values(['date', 'away_lookup']),
        elo_away[['date', 'away_lookup', 'away_elo']].sort_values(['date', 'away_lookup']),
        on='date',
        by='away_lookup',
        direction='backward',
        allow_exact_matches=False,
    )

    merged = merged.dropna(subset=['home_elo', 'away_elo']).copy()
    merged['elo_diff'] = merged['home_elo'] - merged['away_elo']
    merged['elo_sum'] = merged['home_elo'] + merged['away_elo']
    merged['outcome'] = np.select(
        [merged['home_score'] > merged['away_score'], merged['home_score'] == merged['away_score']],
        ['H', 'D'],
        default='A'
    )

    return merged.sort_values('date').reset_index(drop=True)


def train_model(train_df):
    features = ['home_elo', 'away_elo', 'elo_diff', 'elo_sum', 'neutral']
    X = train_df[features]
    y = train_df['outcome']

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', Pipeline(steps=[
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())
            ]), features)
        ]
    )

    model = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', LogisticRegression(max_iter=1000, random_state=SEED))
    ])

    model.fit(X, y)
    return model


def evaluate_model(model, train_df):
    split_idx = int(len(train_df) * 0.8)
    test_df = train_df.iloc[split_idx:].copy()
    X_test = test_df[['home_elo', 'away_elo', 'elo_diff', 'elo_sum', 'neutral']]
    y_test = test_df['outcome']
    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)
    classes = model.named_steps['classifier'].classes_

    accuracy = float((preds == y_test).mean())
    y_index = pd.Categorical(y_test, categories=classes).codes
    chosen = probs[np.arange(len(probs)), y_index]
    log_loss = float(-np.mean(np.log(np.clip(chosen, 1e-12, 1.0))))
    return {'accuracy': accuracy, 'log_loss': log_loss}


def ensure_team_ratings(latest_elo):
    lookup = build_name_lookup(latest_elo['team'].tolist())
    ratings = {}
    for requested_team in TEAM_CONFEDS:
        source_name = TEAM_NAME_ALIASES.get(requested_team, requested_team)
        actual_name = lookup.get(source_name)
        if actual_name is None:
            raise ValueError(f'Missing Elo rating for {requested_team}')
        ratings[requested_team] = float(latest_elo.loc[latest_elo['team'] == actual_name, 'elo'].iloc[0])
    return ratings


In [ ]:
def make_feature_frame(home_team, away_team, team_ratings, neutral=True):
    home_elo = team_ratings[home_team]
    away_elo = team_ratings[away_team]
    return pd.DataFrame([{
        'home_elo': home_elo,
        'away_elo': away_elo,
        'elo_diff': home_elo - away_elo,
        'elo_sum': home_elo + away_elo,
        'neutral': int(neutral),
    }])


def predict_outcome_probs(model, home_team, away_team, team_ratings):
    frame = make_feature_frame(home_team, away_team, team_ratings, neutral=True)
    probs = model.predict_proba(frame)[0]
    classes = model.named_steps['classifier'].classes_
    return {cls: float(prob) for cls, prob in zip(classes, probs)}


def sample_scoreline(outcome, rng):
    if outcome == 'D':
        goals = int(rng.choice([0, 1, 2, 3], p=[0.24, 0.51, 0.20, 0.05]))
        return goals, goals

    loser_goals = int(rng.choice([0, 1, 2, 3], p=[0.48, 0.33, 0.14, 0.05]))
    margin = int(rng.choice([1, 2, 3, 4], p=[0.58, 0.26, 0.11, 0.05]))
    winner_goals = min(loser_goals + margin, 6)
    if outcome == 'H':
        return winner_goals, loser_goals
    return loser_goals, winner_goals


def simulate_match(model, home_team, away_team, team_ratings, rng, knockout=False):
    probs = predict_outcome_probs(model, home_team, away_team, team_ratings)
    outcome = str(rng.choice(['H', 'D', 'A'], p=[probs['H'], probs['D'], probs['A']]))
    home_goals, away_goals = sample_scoreline(outcome, rng)

    winner = None
    if home_goals > away_goals:
        winner = home_team
    elif away_goals > home_goals:
        winner = away_team
    elif knockout:
        non_draw_total = probs['H'] + probs['A']
        if non_draw_total == 0:
            winner = str(rng.choice([home_team, away_team]))
        else:
            shootout_prob = probs['H'] / non_draw_total
            winner = home_team if rng.random() < shootout_prob else away_team

    return MatchResult(home_team, away_team, home_goals, away_goals, winner)


def create_pots(team_ratings):
    ordered = sorted(team_ratings, key=lambda team: team_ratings[team], reverse=True)
    non_hosts = [team for team in ordered if team not in HOSTS]
    pot1 = HOSTS + non_hosts[:9]
    remaining = [team for team in non_hosts if team not in pot1]
    pots = [pot1]
    for start in range(0, len(remaining), 12):
        pots.append(remaining[start:start + 12])
    return pots


def confed_limit(confed):
    return 2 if confed == 'UEFA' else 1


def is_valid_group_addition(group, team):
    confed = TEAM_CONFEDS[team]
    return len(group) < 4 and sum(1 for member in group if TEAM_CONFEDS[member] == confed) < confed_limit(confed)


def draw_groups(team_ratings, rng):
    pots = create_pots(team_ratings)
    groups = {group: [] for group in GROUP_NAMES}
    groups['A'].append('United States')
    groups['B'].append('Mexico')
    groups['C'].append('Canada')

    pot1_others = [team for team in pots[0] if team not in HOSTS]
    rng.shuffle(pot1_others)
    for group_name, team in zip(GROUP_NAMES[3:], pot1_others):
        groups[group_name].append(team)

    remaining_assignments = []
    for pot in [pots[1], pots[2], pots[3], pots[0]]:
        candidates = [team for team in pot if team not in HOSTS]
        if pot is pots[0]:
            candidates = [team for team in candidates if team not in pot1_others]
        rng.shuffle(candidates)
        for team in candidates:
            remaining_assignments.append(team)

    def backtrack(index):
        if index == len(remaining_assignments):
            return all(len(members) == 4 for members in groups.values())

        team = remaining_assignments[index]
        target_groups = sorted(GROUP_NAMES, key=lambda group_name: (len(groups[group_name]), rng.random()))
        for group_name in target_groups:
            if not is_valid_group_addition(groups[group_name], team):
                continue
            groups[group_name].append(team)
            if backtrack(index + 1):
                return True
            groups[group_name].pop()
        return False

    if not backtrack(0):
        raise RuntimeError('Could not generate a valid 2026 group draw')
    return groups


In [ ]:
def build_group_table(group_teams, matches, team_ratings):
    rows = {
        team: {'team': team, 'pts': 0, 'gf': 0, 'ga': 0, 'gd': 0, 'wins': 0, 'elo': team_ratings[team]}
        for team in group_teams
    }

    for match in matches:
        rows[match.home_team]['gf'] += match.home_goals
        rows[match.home_team]['ga'] += match.away_goals
        rows[match.away_team]['gf'] += match.away_goals
        rows[match.away_team]['ga'] += match.home_goals

        if match.home_goals > match.away_goals:
            rows[match.home_team]['pts'] += 3
            rows[match.home_team]['wins'] += 1
        elif match.away_goals > match.home_goals:
            rows[match.away_team]['pts'] += 3
            rows[match.away_team]['wins'] += 1
        else:
            rows[match.home_team]['pts'] += 1
            rows[match.away_team]['pts'] += 1

    for team in rows:
        rows[team]['gd'] = rows[team]['gf'] - rows[team]['ga']

    table = pd.DataFrame(rows.values()).sort_values(
        ['pts', 'gd', 'gf', 'wins', 'elo'],
        ascending=[False, False, False, False, False]
    )
    table['rank'] = range(1, len(table) + 1)
    return table[['rank', 'team', 'pts', 'gd', 'gf', 'ga', 'wins']].reset_index(drop=True)


def simulate_group_stage(model, groups, team_ratings, rng):
    group_tables = {}
    all_matches = []

    for group_name, teams in groups.items():
        matches = []
        for home_team, away_team in combinations(teams, 2):
            match = simulate_match(model, home_team, away_team, team_ratings, rng, knockout=False)
            matches.append(match)
            all_matches.append(match)

        table = build_group_table(teams, matches, team_ratings)
        table.insert(0, 'group', group_name)
        group_tables[group_name] = table

    return group_tables, all_matches


def get_advancing_teams(group_tables, team_ratings):
    winners = []
    runners_up = []
    third_place_rows = []

    for group_name in GROUP_NAMES:
        table = group_tables[group_name]
        winners.append(str(table.iloc[0]['team']))
        runners_up.append(str(table.iloc[1]['team']))
        third_place_rows.append({
            'group': group_name,
            'team': str(table.iloc[2]['team']),
            'pts': int(table.iloc[2]['pts']),
            'gd': int(table.iloc[2]['gd']),
            'gf': int(table.iloc[2]['gf']),
            'wins': int(table.iloc[2]['wins']),
            'elo': team_ratings[str(table.iloc[2]['team'])],
        })

    third_df = pd.DataFrame(third_place_rows).sort_values(
        ['pts', 'gd', 'gf', 'wins', 'elo'],
        ascending=[False, False, False, False, False]
    )
    best_thirds = third_df.head(8)['team'].tolist()
    return winners, runners_up, best_thirds


def rank_knockout_seeds(group_tables, winners, runners_up, best_thirds, team_ratings):
    rows = []
    for group_name in GROUP_NAMES:
        table = group_tables[group_name]
        for _, row in table.iterrows():
            team = str(row['team'])
            if team not in winners and team not in runners_up and team not in best_thirds:
                continue
            band = 0 if team in winners else 1 if team in runners_up else 2
            rows.append({
                'team': team,
                'band': band,
                'pts': int(row['pts']),
                'gd': int(row['gd']),
                'gf': int(row['gf']),
                'wins': int(row['wins']),
                'elo': team_ratings[team],
            })

    seeds = pd.DataFrame(rows).sort_values(
        ['band', 'pts', 'gd', 'gf', 'wins', 'elo'],
        ascending=[True, False, False, False, False, False]
    )
    return seeds['team'].tolist()


def play_knockout_round(teams, model, team_ratings, rng):
    winners = []
    matches = []
    for index in range(len(teams) // 2):
        home_team = teams[index]
        away_team = teams[-(index + 1)]
        match = simulate_match(model, home_team, away_team, team_ratings, rng, knockout=True)
        winners.append(match.winner or home_team)
        matches.append(match)
    return winners, matches


def simulate_tournament(model, team_ratings, seed=SEED):
    rng = np.random.default_rng(seed)
    groups = draw_groups(team_ratings, rng)
    group_tables, group_matches = simulate_group_stage(model, groups, team_ratings, rng)
    winners, runners_up, best_thirds = get_advancing_teams(group_tables, team_ratings)
    seeds = rank_knockout_seeds(group_tables, winners, runners_up, best_thirds, team_ratings)

    round_of_16_teams, round_of_32_matches = play_knockout_round(seeds, model, team_ratings, rng)
    quarterfinal_teams, round_of_16_matches = play_knockout_round(round_of_16_teams, model, team_ratings, rng)
    semifinal_teams, quarterfinal_matches = play_knockout_round(quarterfinal_teams, model, team_ratings, rng)
    finalists, semifinal_matches = play_knockout_round(semifinal_teams, model, team_ratings, rng)
    champion_list, final_matches = play_knockout_round(finalists, model, team_ratings, rng)

    return {
        'groups': groups,
        'group_tables': group_tables,
        'group_matches': group_matches,
        'round_of_32': round_of_32_matches,
        'round_of_16': round_of_16_matches,
        'quarterfinals': quarterfinal_matches,
        'semifinals': semifinal_matches,
        'final': final_matches,
        'champion': champion_list[0],
        'best_thirds': best_thirds,
    }


def monte_carlo_champions(model, team_ratings, runs=SIMULATION_RUNS):
    champions = []
    for sim_seed in range(SEED, SEED + runs):
        champions.append(simulate_tournament(model, team_ratings, sim_seed)['champion'])

    counts = pd.Series(champions).value_counts().rename_axis('team').reset_index(name='titles')
    counts['title_pct'] = 100 * counts['titles'] / runs
    return counts


def format_match(match):
    score = f"{match.home_team} {match.home_goals}-{match.away_goals} {match.away_team}"
    if match.winner and match.home_goals == match.away_goals:
        score += f" ({match.winner} win on pens)"
    return score


## Load Data And Train Model


In [ ]:
results_df, elo_df = load_data()
latest_elo = get_latest_elo(elo_df)
team_ratings = ensure_team_ratings(latest_elo)
training_df = prepare_training_data(results_df, elo_df)
model = train_model(training_df)
metrics = evaluate_model(model, training_df)

summary = pd.DataFrame({
    'metric': ['training_rows', 'holdout_accuracy', 'holdout_log_loss', 'latest_elo_snapshot'],
    'value': [len(training_df), round(metrics['accuracy'], 3), round(metrics['log_loss'], 3), str(latest_elo['date'].max().date())]
})
summary


## One Tournament Simulation


In [ ]:
simulation = simulate_tournament(model, team_ratings, seed=SEED)
simulation['champion']


In [ ]:
group_draw_df = pd.DataFrame([
    {'group': group_name, 'teams': ', '.join(simulation['groups'][group_name])}
    for group_name in GROUP_NAMES
])
group_draw_df


In [ ]:
for group_name in GROUP_NAMES:
    print(f'Group {group_name}')
    display(simulation['group_tables'][group_name])


In [ ]:
pd.DataFrame({'best_third_placed_teams': simulation['best_thirds']})


In [ ]:
for round_name in ['round_of_32', 'round_of_16', 'quarterfinals', 'semifinals', 'final']:
    print(round_name.replace('_', ' ').title())
    display(pd.DataFrame({'match': [format_match(match) for match in simulation[round_name]]}))


## Title Odds Across Many Simulations


In [ ]:
title_odds = monte_carlo_champions(model, team_ratings, runs=SIMULATION_RUNS)
title_odds.head(20)


In [ ]:
title_odds.plot.bar(x='team', y='title_pct', figsize=(12, 5), legend=False, title='2026 World Cup title odds (%)')
